# Agent v5 -- Interactive Test Notebook

Tests the v5 agent end-to-end against **real data** in the `enriched_property_listing` MongoDB collection (schema_version 1): tools, streaming, multi-turn memory, and structured output.

**Prerequisites:** `.env` with `AI_GATEWAY_API_KEY`, `MONGODB_PW` (v5 does not use Upstash)

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Root:', ROOT)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(ROOT / '.env')

for var in ['AI_GATEWAY_API_KEY', 'MONGODB_PW']:
    status = 'OK' if os.getenv(var) else 'MISSING'
    print(f'{var}: {status}')

## 1. Real data sanity check -- `enriched_property_listing`

In [ ]:
from utility.property_listing_init import get_enriched_property_listing_collections

col = get_enriched_property_listing_collections()
total = col.count_documents({})
active = col.count_documents({'listing_status': 'active'})
print(f'documents: {total} total / {active} active')
print('schema_version values:', col.distinct('schema_version'))
print('offer types:', col.distinct('offer.offer_type'))
print('categories:', col.distinct('main_category'))

In [ ]:
from agent.v5.tools._utils import serialize_listing

doc = col.find_one({'listing_status': 'active', 'traits.industrial': {'$exists': True}})
s = serialize_listing(doc)
for k, v in s.items():
    print(f'{k:24} {v}')

In [ ]:
import json
from agent.v5.tools._utils import serialize_chat_listing

# Frontend ChatListingCard shape — exactly what the property_cards SSE event carries
# (industrialprop_FE app/landy-ai/types.ts ChatListing)
card = serialize_chat_listing(doc)
print(json.dumps({**card, 'images': f"<{len(card['images'])} images>"}, indent=2, default=str))

## 2. Direct tool test -- `find_listings` (real MongoDB query)

Outside a LangGraph run there is no stream writer, so we capture SSE events by patching `get_stream_writer` in the tool module.

In [ ]:
import contextlib

@contextlib.contextmanager
def capture_events(*modules):
    """Temporarily replace get_stream_writer in tool modules with an event collector."""
    events = []
    originals = {m: m.get_stream_writer for m in modules}
    for m in modules:
        m.get_stream_writer = lambda: events.append
    try:
        yield events
    finally:
        for m, orig in originals.items():
            m.get_stream_writer = orig

In [ ]:
import agent.v5.tools.find_listings as fl

with capture_events(fl) as events:
    result = await fl.find_listings.ainvoke({
        'offer_type': 'rent',
        'property_category': ['factory', 'warehouse'],
        'region': 'Selangor',
    })

print(f"total_found: {result['total_found']}")
print(f"filters_applied: {result['filters_applied']}")
print(f"location_breakdown: {result['location_breakdown']}")
print()
for r in result['property_listing_result'][:5]:
    print(f"  [{r['property_id']}] {r['title']}")
    print(f"      {r['offer_type']} | RM{r['price']:,} | {r['city']} | features: {r['extracted_key_features'][:2]}")
print()
print('SSE events:', [e['event'] for e in events])

# property_cards carries the frontend ChatListing shape, not the LLM-facing flat shape
cards_event = next(e for e in events if e['event'] == 'property_cards')
first_card = cards_event['listings'][0]
print('card keys:    ', sorted(first_card.keys()))
print('card location:', first_card['location'])
print('card specs:   ', first_card['specifications'])

In [ ]:
# Proximity filter on real data: factories within 5km of a highway
with capture_events(fl) as events:
    near_highway = await fl.find_listings.ainvoke({
        'property_category': ['factory'],
        'max_highway_km': 5.0,
    })

print(f"total_found: {near_highway['total_found']}")
for r in near_highway['property_listing_result'][:5]:
    hw = r['nearest_highway'] or {}
    print(f"  [{r['property_id']}] {r['title'][:60]} -- {hw.get('name')} @ {hw.get('distance_km')}km")

## 3. Direct tool test -- `get_listing_detail`

In [ ]:
import agent.v5.tools.get_listing_detail as gld

pid = result['property_listing_result'][0]['property_id']

with capture_events(gld) as events:
    detail = await gld.get_listing_detail.ainvoke({'property_id': pid})

print(f"Title:        {detail['title']}")
print(f"Description:  {(detail['description'] or '')[:200]}...")
print(f"Power supply: {detail['power_supply']}")
print(f"Loading bays: {detail['loading_bays']}")
print(f"Risk factors: {detail['risk_factors']}")
print(f"Similar IDs:  {detail['similar_listing_id']}")
print(f"Images:       {len(detail['images'])}")
print()
print('SSE events:', [e['event'] for e in events])

## 4. Create agent (InMemorySaver -- no Postgres needed)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from agent.v5.orchestration import create_agent, extract_v5_state

checkpointer = InMemorySaver()
agent = create_agent(checkpointer)

def unwrap(resp):
    """langgraph >= 1.1 returns GraphOutput; the result dict lives on .value"""
    return getattr(resp, 'value', resp)

def final_answer(resp):
    """Last non-empty AI text message — the user-facing answer for the turn."""
    for m in reversed(unwrap(resp)['messages']):
        if str(getattr(m, 'type', '')) == 'ai':
            c = m.content
            if isinstance(c, list):
                c = ''.join(b.get('text', '') if isinstance(b, dict) else str(b) for b in c)
            if c:
                return c
    return ''

print('Agent created:', type(agent))

## 5. Single-turn invoke

In [ ]:
THREAD_ID = 'v5-test-01'
QUERY = 'factory for rent in Selangor under RM200k per month'

response = await agent.ainvoke(
    {'messages': QUERY},
    {'configurable': {'thread_id': THREAD_ID}},
    version='v2',
)

answer = final_answer(response)
print('=== answer ===')
print(answer)
print()

# chips/CTA come from a separate forced structured-output call (two-phase design)
s = await extract_v5_state(QUERY, answer)
print('follow_up_chips:   ', s.follow_up_chips)
print('live_agent_cta:    ', s.live_agent_cta)
print('live_agent_trigger:', s.live_agent_trigger)

## 6. Multi-turn conversation (filter persistence)

In [ ]:
THREAD_ID_MT = 'v5-test-multi-turn-01'

turns = [
    'warehouse for rent in Selangor',
    'only ones near a highway please',
    'which of these has the highest ceiling?',
]

for i, msg in enumerate(turns, 1):
    resp = await agent.ainvoke(
        {'messages': msg},
        {'configurable': {'thread_id': THREAD_ID_MT}},
        version='v2',
    )
    print(f'--- Turn {i}: {msg!r} ---')
    print(final_answer(resp)[:400])
    print()

## 7. Streaming -- watch SSE events live

Mirrors `/api/v5/stream` behaviour: only AI answer tokens are printed (tool-result chunks and the structured-output acknowledgement are filtered out, exactly like the endpoint does). Note: the `thread_id` first-frame event is added by the API endpoint itself, so it does not appear when streaming the agent directly.

In [ ]:
THREAD_ID_STREAM = 'v5-test-stream-03'

async for raw_event in agent.astream(
    {'messages': 'newest warehouses for sale in Selangor'},
    {'configurable': {'thread_id': THREAD_ID_STREAM}},
    stream_mode=['updates', 'messages', 'custom'],
    subgraphs=True,
    version='v2',
):
    # langgraph >= 1.1 yields dicts; older versions yield 3-tuples
    if isinstance(raw_event, dict):
        type_, ns, data = raw_event.get('type'), raw_event.get('ns'), raw_event.get('data')
    elif isinstance(raw_event, tuple) and len(raw_event) == 3:
        type_, ns, data = raw_event
    else:
        continue

    if type_ == 'custom':
        event_name = data.get('event', '?') if isinstance(data, dict) else '?'
        if event_name == 'search_start':
            print(f"\n[search_start] filters_active={data.get('filters_active')}")
        elif event_name == 'property_cards':
            print(f"[property_cards] {len(data.get('listings', []))} frontend-ready cards")
        elif event_name == 'search_complete':
            print(f"[search_complete] total_found={data.get('total_found')}")
        else:
            print(f'[custom] {event_name}')

    elif type_ == 'messages':
        if isinstance(data, tuple) and data:
            msg = data[0]
            # mirror /api/v5/stream: only AI answer tokens reach the client
            mtype = str(getattr(msg, 'type', ''))
            if not (mtype == 'ai' or mtype.startswith('AIMessage')):
                continue
            token = getattr(msg, 'content', '')
            if isinstance(token, list):
                token = ''.join(b.get('text', '') if isinstance(b, dict) else str(b) for b in token)
            if token:
                print(token, end='', flush=True)

print('\n--- stream complete ---')

## 8. Live agent CTA trigger test (transact_intent)

In [ ]:
THREAD_ID_CTA = 'v5-test-cta-01'
CTA_QUERY = 'I want to book a viewing and make an offer on a factory in Shah Alam'

resp = await agent.ainvoke(
    {'messages': CTA_QUERY},
    {'configurable': {'thread_id': THREAD_ID_CTA}},
    version='v2',
)

answer = final_answer(resp)
print(answer[:400])
print()
s = await extract_v5_state(CTA_QUERY, answer)
print('live_agent_cta:    ', s.live_agent_cta, '(expected: True)')
print('live_agent_trigger:', s.live_agent_trigger, '(expected: transact_intent)')
print('follow_up_chips:   ', s.follow_up_chips)

## 9. Response scope -- domain-adjacent question (no search expected)

The tiered scope policy should answer property-knowledge questions conversationally WITHOUT calling `find_listings`, and decline unrelated requests warmly without the old canned script.

In [ ]:
THREAD_ID_SCOPE = 'v5-test-scope-01'
SCOPE_QUERY = 'what does floor loading 29 kN/m2 actually mean? is it enough for CNC machines?'

resp = await agent.ainvoke(
    {'messages': SCOPE_QUERY},
    {'configurable': {'thread_id': THREAD_ID_SCOPE}},
    version='v2',
)

out = unwrap(resp)
called_find_listings = any(
    tc.get('name') == 'find_listings'
    for m in out['messages']
    for tc in (getattr(m, 'tool_calls', None) or [])
)

print(final_answer(resp)[:500])
print()
print('find_listings called:', called_find_listings, '(expected: False)')
s = await extract_v5_state(SCOPE_QUERY, final_answer(resp))
print('chips:', s.follow_up_chips)

In [ ]:
# Unrelated request — expect a brief warm redirect (no canned script), no search
resp = await agent.ainvoke(
    {'messages': 'can you help me write a python script for web scraping?'},
    {'configurable': {'thread_id': THREAD_ID_SCOPE}},
    version='v2',
)

answer = final_answer(resp)
print(answer[:400])
print()
print('old canned script used:', 'I can only help with industrial property searches' in answer, '(expected: False)')

## 10. Postgres checkpointer (production mode)

Uncomment and set `DB_URI` to test with persistent memory across kernel restarts.

In [ ]:
# DB_URI = os.getenv('DB_URI')

# from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver

# async with AsyncPostgresSaver.from_conn_string(DB_URI) as cp:
#     await cp.setup()
#     pg_agent = create_agent(cp)

#     resp = await pg_agent.ainvoke(
#         {'messages': 'warehouse for sale in Klang'},
#         {'configurable': {'thread_id': 'v5-prod-test-01'}},
#         version='v2',
#     )
#     print(final_answer(resp))